# 使用 MobileNetV2 进行迁移学习

欢迎来到本周的作业，你将在预训练 CNN 上使用迁移学习来构建羊驼/非羊驼分类器！

<img src="images/alpaca.png" style="width:300px;height:220px;">

预训练模型是在大型数据集上训练并保存的网络，可以帮助你高效、低成本地构建自己的模型。本次使用的 MobileNetV2 强调运行速度和计算效率，已在 ImageNet 上预训练；该数据集包含超过 1400 万张图像和 1000 个类别。

在本作业结束时，你将能够：

- 从目录创建数据集
- 使用 Sequential API 预处理和扩充数据
- 让预训练模型适应新数据并使用 Functional API 和 MobileNet 训练分类器
- 微调分类器的最终层以提高准确性

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/convolutional-neural-networks/supplement/DS4yP/h-ow-to-refresh-your-workspace) 获取一份新的作业。

## 目录

- [1 - 依赖库](#1)
    - [1.1 创建数据集并将其拆分为训练集和验证集](#1-1)
- [2 - 预处理和增强训练数据](#2)
    - [练习 1 - data_augmenter](#ex-1)
- [3 - 使用 MobileNetV2 进行迁移学习](#3)
    - [3.1 - MobileNetV2 卷积块的内部结构](#3-1)
    - [3.2 - 使用 Functional API 冻结层](#3-2)
        - [练习 2 - alpaca_model](#ex-2)
    - [3.3 - 微调模型](#3-3)
        - [练习 3](#ex-3)

<a name='1'></a>
## 1 - 依赖库

In [ ]:
### v2.1

In [ ]:
import matplotlib.pyplot as plt
import json
import numpy as np
import os
import tensorflow as tf
import tensorflow.keras.layers as tfl

from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.layers.experimental.preprocessing import RandomFlip, RandomRotation

<a name='1-1'></a>
### 1.1 创建数据集并将其拆分为训练集和验证集

在 Keras 中训练和评估深度学习模型时，从存储在磁盘上的图像文件生成数据集既简单又快速。调用 `image_data_set_from_directory()` 从目录中读取并创建训练和验证数据集。 

如果你指定验证分割，你还需要指定每个部分的子集。只需将训练集设置为 `subset='training'`，将验证集设置为 `subset='validation'`。

你还可以将种子设置为相互匹配，这样你的训练集和验证集就不会重叠。 :) 

In [ ]:
BATCH_SIZE = 32
IMG_SIZE = (160, 160)
directory = "dataset/"
train_dataset = image_dataset_from_directory(directory,
                                             shuffle=True,
                                             batch_size=BATCH_SIZE,
                                             image_size=IMG_SIZE,
                                             validation_split=0.2,
                                             subset='training',
                                             seed=42)
validation_dataset = image_dataset_from_directory(directory,
                                             shuffle=True,
                                             batch_size=BATCH_SIZE,
                                             image_size=IMG_SIZE,
                                             validation_split=0.2,
                                             subset='validation',
                                             seed=42)

现在让我们看一下训练集的一些图像： 

**注意：** 原始数据集中也有一些标签错误的图像。

In [ ]:
class_names = train_dataset.class_names

plt.figure(figsize=(10, 10))
for images, labels in train_dataset.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")

<a name='2'></a>
## 2 - 预处理和增强训练数据

你可能在之前的 TensorFlow 作业中遇到过 `dataset.prefetch`，这是数据预处理中重要的额外步骤。 

使用 `prefetch()` 可以防止从磁盘读取时可能出现的内存瓶颈。它通过根据输入数据创建源数据集、应用转换对其进行预处理，然后一次迭代数据集一个元素，预留一些数据并使其在需要时做好准备。由于迭代是流式的，因此数据不需要装入内存。

你可以手动设置要预取的元素数量，也可以使用 `tf.data.experimental.AUTOTUNE` 自动选择参数。自动调谐提示 `tf.data` 在运行时动态调整该值，方法是跟踪每个操作所花费的时间并将这些时间输入优化算法。优化算法尝试在所有可调操作中找到 CPU 预算的最佳分配。 

为了增加训练集的多样性并帮助你的模型更好地学习数据，标准做法是通过变换 即、随机翻转和旋转图像来增强图像。 Keras 的Sequential API 为此类数据增强提供了一种简单的方法，具有内置的、可定制的预处理层。这些层与模型的其余部分一起保存，并且可以在以后重复使用。  啊啊，太方便了！ 

一如既往，我们邀请你阅读官方文档，你可以找到有关数据增强的 [这里](https://www.tensorflow.org/tutorials/images/data_augmentation)。


In [ ]:
AUTOTUNE = tf.data.experimental.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)

<a name='ex-1'></a>
### 练习 1 - data_augmenter

实现函数用于数据增强。使用由 2 层组成的 `Sequential` Keras 模型：
* `RandomFlip('horizontal')`
* `RandomRotation(0.2)`

In [ ]:
# UNQ_C1
# GRADED FUNCTION: data_augmenter
def data_augmenter():
    '''
    创建一个由 2 层组成的序列模型
    返回：
        tf.keras.Sequential
    '''
    ### START CODE HERE
    data_augmentation = None
    data_augmentation.add(None)
    data_augmentation.add(None)
    ### END CODE HERE
    
    return data_augmentation

In [ ]:
augmenter = data_augmenter()

assert(augmenter.layers[0].name.startswith('random_flip')), "First layer must be RandomFlip"
assert augmenter.layers[0].mode == 'horizontal', "RadomFlip parameter must be horizontal"
assert(augmenter.layers[1].name.startswith('random_rotation')), "Second layer must be RandomRotation"
assert augmenter.layers[1].factor == 0.2, "Rotation factor must be 0.2"
assert len(augmenter.layers) == 2, "The model must have only 2 layers"

print('\033[92mAll tests passed!')


看一下训练集中的图像如何通过简单的变换进行增强：

从一种可爱的动物到该可爱动物的 9 种变体，只需三行代码。现在你的模型有更多值得学习的地方。

In [ ]:
data_augmentation = data_augmenter()

for image, _ in train_dataset.take(1):
    plt.figure(figsize=(10, 10))
    first_image = image[0]
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        augmented_image = data_augmentation(tf.expand_dims(first_image, 0))
        plt.imshow(augmented_image[0] / 255)
        plt.axis('off')

接下来，使用 TensorFlow 中 MobileNet 提供的预处理工具归一化输入。预训练模型使用取值范围为 [-1,1] 的输入，因此应调用 tf.keras.applications.mobilenet_v2.preprocess_input，使输入与预训练时采用的标准一致。

<div style="color:blue;">

**你应该记住的是：**

* 调用image_data_set_from_directory()时，指定train/val子集并匹配种子以防止重叠
* 使用 prefetch() 防止从磁盘读取时出现内存瓶颈
* 通过简单的数据增强（例如旋转和翻转）为你的模型提供更多学习机会。
* 使用预训练模型时，最好重复使用其训练时使用的权重。

</div>

In [ ]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

<a name='3'></a>
## 3 - 使用 MobileNetV2 进行迁移学习 

MobileNetV2 在 ImageNet 上进行训练，并针对在移动和其他低功耗应用程序上运行进行了优化。它有 155 层深（以防万一你想自己绘制模型，为长途旅行做好准备！）并且对于目标检测和图像分割任务以及像这样的分类任务非常有效。该架构具有三个定义特征：

*   深度可分离卷积
*   层间的低维输入和输出瓶颈
*   瓶颈层之间的快捷连接

<a name='3-1'></a>
### 3.1 - MobileNetV2 卷积块的内部结构

MobileNetV2 使用深度可分离卷积作为高效的构建块。传统的卷积通常非常消耗资源，而深度可分离卷积能够减少可训练参数和操作的数量，并且还可以通过两步加速卷积： 

1. 第一步通过独立地对每个通道进行卷积来计算中间结果。这就是深度卷积。

2. 第二步使用逐点卷积（pointwise convolution），对第一步的各通道输出进行组合。每个 1x1 卷积核在一个空间位置上汇总各通道的信息，得到一个输出值；多个卷积核生成多个输出通道。输出的**空间尺寸与逐通道卷积（depthwise convolution）一致，通道数等于逐点卷积的卷积核数量**。

<img src="images/mobilenetv2.png" style="width:650px;height:450px;">
<div style="text-align:center;color:purple;"><u>  <b>图 1</b> </u> : <b>MobileNetV2 Architecture</b> <br> 该图的灵感来自原始看到的 <a href="https://ai.googleblog.com/2018/04/mobilenetv2-next-generation-of-on.html#:~:text=MobileNetV2%20is%20a%20significant%20improvement,object%20detection%20and%20semantic%20segmentation.">here</a>.</div>

每个块都由一个倒置的残差结构组成，两端都有一个瓶颈。这些瓶颈将中间输入和输出编码在低维空间中，并防止非线性破坏重要信息。 

快捷连接与传统残差网络中的快捷连接类似，具有加快训练速度和改进预测的相同目的。这些连接跳过中间卷积并连接瓶颈层。 

让我们尝试使用预训练模型中的所有层来训练你的基本模型。 

与你重用 MobileNetV2 训练时使用的预训练归一化值的方式类似，你还可以从 ImageNet 加载预训练权重。

通过指定 `weights='imagenet'`，为 ImageNet 下载权重。但对于本实验，你将从工作区本地加载它们。

In [ ]:
IMG_SHAPE = IMG_SIZE + (3,)
base_model_path="imagenet_base_model/with_top_mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_160.h5"

base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SHAPE,
                                               include_top=True,
                                               weights=base_model_path)

打印下面的模型摘要，以查看所有模型的层、其输出的形状以及参数总数（可训练和不可训练）。 

In [ ]:
base_model.summary()

注意这里的最后两层。它们是所谓的顶层，负责模型中的分类

In [ ]:
nb_layers = len(base_model.layers)
print(base_model.layers[nb_layers - 2].name)
print(base_model.layers[nb_layers - 1].name)

请注意摘要中的一些层，例如 `Conv2D` 和 `DepthwiseConv2D` 以及它们如何遵循扩展到深度卷积到投影的过程。与 BatchNormalization 和 ReLU 结合，它们构成了前面提到的瓶颈层。

<div style="color:blue;">

**你应该记住什么**：

* MobileNetV2的独特之处在于： 
  * 深度可分离卷积，提供轻量级特征过滤和创建
  * 在块的任一端保留重要信息的输入和输出瓶颈
* 深度可分离卷积处理空间和深度（通道数）维度

</div>

接下来，从 TensorFlow 数据集中选择第一批来使用图像，并通过 MobileNetV2 基本模型运行它以测试对某些图像的预测。 

In [ ]:
image_batch, label_batch = next(iter(train_dataset))
feature_batch = base_model(image_batch)
print(feature_batch.shape)

In [ ]:
#显示一个张量中的不同标签概率 
label_batch

现在解码模型的预测结果。前面打印的批次形状为 (32, 1000)：32 是批量大小，1000 是模型预训练时的类别数。下面基础模型返回的预测结果采用以下格式：

依次为类别编号、可读的类别名称，以及图像属于该类别的概率。每张图像返回两组结果，对应预测概率最高的两个类别。

In [ ]:
base_model.trainable = False
image_var = tf.Variable(preprocess_input(image_batch))
pred = base_model(image_var)

# 函数解码预测
def decode_predictions(preds, top=2):
    results = []
    for pred in preds:
        top_indices = pred.argsort()[-top:][::-1]
        result = [tuple(class_index[str(i)]) + (pred[i],) for i in top_indices]
        results.append(result)
    return results

with open("imagenet_base_model/imagenet_class_index.json", 'r') as f:
    class_index = json.load(f)
    
decoded_predictions = decode_predictions(pred.numpy(), top=2)
print(decoded_predictions)

呃哦。这里有很多标签，其中一些是错误的，但没有一个说“羊驼毛”。

这是因为 ImageNet 上的 MobileNet 预训练没有正确的羊驼标签，所以当你使用完整模型时，你得到的只是一堆错误分类的图像。

幸运的是，你可以删除包含所有分类标签的顶层，并创建一个新的分类层。

<a name='3-2'></a>
### 3.2 - 使用 Functional API 冻结层

<img src="images/snowalpaca.png" style="width:400px;height:250px;">

在接下来的部分中，你将了解如何使用预训练模型来修改分类器任务，以便它能够识别羊驼。你可以通过三个步骤来实现这一目标： 

1. 删除顶层（分类层）
    * 将 `base_model` 中的 `include_top` 设置为 False
2. 添加新的分类器层
    * 通过冻结网络的其余部分来仅训练一层
    * 如前所述，单个神经元足以解决二元分类问题。
3. 冻结基础模型并训练新创建的分类器层
    * 设置 `base model.trainable=False` 以避免更改权重并*仅*训练新层
    * 将 `base_model` 中的训练设置为 False 以避免跟踪批量归一化层中的统计数据

<a name='ex-2'></a>
### 练习 2 - alpaca_model

In [ ]:
# UNQ_C2
# GRADED FUNCTION
def alpaca_model(image_shape=IMG_SIZE, data_augmentation=data_augmenter()):
    ''' 定义 tf.keras 模型，用于 MobileNetV2 模型的二元分类
    参数：
        image_shape -- 图像宽度和高度
        data_augmentation -- 数据增强函数
    返回：
    返回：
        tf.keras.model
    '''
    
    
    input_shape = image_shape + (3,)
    
    ### START CODE HERE
    
    base_model_path="imagenet_base_model/without_top_mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_160_no_top.h5"
    
    base_model = tf.keras.applications.MobileNetV2(input_shape=None,
                                                   include_top=None, # <== 重要！！！
                                                   weights=base_model_path)
    
    # 通过使其不可训练来冻结基本模型
    base_model.trainable = None 

    # 创建输入层（与imageNetv2输入大小相同）
    inputs = tf.keras.Input(shape=None) 
    
    # 将数据增强应用于输入
    x = None
    
    # 使用模型训练时相同的权重进行数据预处理
    x = preprocess_input(None) 
    
    # 将训练设置为 False 以避免在批量归一化层中跟踪统计数据
    x = base_model(None, training=None) 
    
    # 添加新的二元分类层
    # 使用全局平均池化来汇总每个通道中的信息
    x = None()(x) 
    # 以 0.2 的概率包含 Dropout 以避免过拟合
    x = None(None)(x)
        
    # 使用具有一个神经元的预测层（作为二元分类器只需要一个）
    outputs = None
    
    ### END CODE HERE
    
    model = tf.keras.Model(inputs, outputs)
    
    return model

使用之前定义的 data_augmentation 函数创建新模型。

In [ ]:
model2 = alpaca_model(IMG_SIZE, data_augmentation)

In [ ]:
from test_utils import summary, comparator

alpaca_summary = [['InputLayer', [(None, 160, 160, 3)], 0],
                    ['Sequential', (None, 160, 160, 3), 0],
                    ['TensorFlowOpLayer', [(None, 160, 160, 3)], 0],
                    ['TensorFlowOpLayer', [(None, 160, 160, 3)], 0],
                    ['Functional', (None, 5, 5, 1280), 2257984],
                    ['GlobalAveragePooling2D', (None, 1280), 0],
                    ['Dropout', (None, 1280), 0, 0.2],
                    ['Dense', (None, 1), 1281, 'linear']] #线性默认为激活值

comparator(summary(model2), alpaca_summary)

for layer in summary(model2):
    print(layer)
    

基础学习率已经为你设置好了，因此你可以继续编译新模型并运行 5 个 epoch：

In [ ]:
base_learning_rate = 0.001
model2.compile(optimizer=tf.keras.optimizers.Adam(lr=base_learning_rate),
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
initial_epochs = 5
history = model2.fit(train_dataset, validation_data=validation_dataset, epochs=initial_epochs)

绘制训练和验证准确性图：

In [ ]:
acc = [0.] + history.history['accuracy']
val_acc = [0.] + history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([min(plt.ylim()),1])
plt.title('Training and Validation Accuracy')

plt.subplot(2, 1, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0,1.0])
plt.title('Training and Validation Loss')
plt.xlabel('epoch')
plt.show()

In [ ]:
class_names

结果还不错，但还可以更好。接下来，尝试一些微调。

<a name='3-3'></a>
### 3.3 - 微调模型

你可以通过在最后几层重新运行优化器来尝试微调模型，以提高准确性。当你使用较小的学习率时，你可以采取较小的步骤来使其更接近新数据。在迁移学习中，实现这一目标的方法是解冻网络末端的层，然后使用非常低的学习率在最后层上重新训练模型。调整你的学习率以更小的步骤遍历这些层可以产生更精细的细节 - 和更高的精度。

可以这样理解：网络浅层学习边缘等低层特征，深层则逐渐提取细毛、尖耳等更复杂的高层特征。在迁移学习中，低层特征通常适用于多种图像，因此可以保持这些层的参数不变。对于新数据，通常希望高层特征作出调整，让网络学会检测与新任务更相关的特征，例如柔软的皮毛或大牙齿。

要实现这一点，只需解冻最后的层并使用较小的学习率重新运行优化器，同时保持所有其他层冻结。

最终层的实际开始位置有点随意，所以请随意调整这个数字。重要的一点是，后面的层是网络的一部分，其中包含针对你的问题更具体的细节（尖尖的耳朵、毛茸茸的尾巴）。

首先，通过设置 `base_model.trainable=True` 解冻基础模型，设置一个层进行微调，然后重新冻结之前的所有层。再运行几个epoch，看看你的准确性是否有所提高！

<a name='ex-3'></a>
### 练习 3

In [ ]:
# UNQ_C3
base_model = model2.layers[4]
base_model.trainable = True
# 我们来看看基础模型有多少层
print("Number of layers in the base model: ", len(base_model.layers))

# 从这一层开始微调
fine_tune_at = 120

### START CODE HERE

# 冻结 `fine_tune_at` 层之前的所有层
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = None
    
# 定义二元交叉熵损失函数。使用 from_logits=True
loss_function=None
# 定义 Adam 优化器，其学习率为 0.1 * base_learning_rate
optimizer = None
# 使用准确性作为评估指标
metrics=None

### END CODE HERE

model2.compile(loss=loss_function,
              optimizer = optimizer,
              metrics=metrics)

In [ ]:
assert type(loss_function) == tf.python.keras.losses.BinaryCrossentropy, "Not the correct layer"
assert loss_function.from_logits, "Use from_logits=True"
assert type(optimizer) == tf.keras.optimizers.Adam, "This is not an Adam optimizer"
assert optimizer.lr == base_learning_rate / 10, "Wrong learning rate"
assert metrics[0] == 'accuracy', "Wrong metric"

print('\033[92mAll tests passed!')


In [ ]:
fine_tune_epochs = 5
total_epochs =  initial_epochs + fine_tune_epochs

history_fine = model2.fit(train_dataset,
                         epochs=total_epochs,
                         initial_epoch=history.epoch[-1],
                         validation_data=validation_dataset)

啊啊，进步很大！一点点微调确实可以起到很大的作用。

In [ ]:
acc += history_fine.history['accuracy']
val_acc += history_fine.history['val_accuracy']

loss += history_fine.history['loss']
val_loss += history_fine.history['val_loss']

In [ ]:
plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.ylim([0, 1])
plt.plot([initial_epochs-1,initial_epochs-1],
          plt.ylim(), label='Start Fine Tuning')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(2, 1, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.ylim([0, 1.0])
plt.plot([initial_epochs-1,initial_epochs-1],
         plt.ylim(), label='Start Fine Tuning')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.xlabel('epoch')
plt.show()

<div style="color:blue;">

**你应该记住什么**：

* 要使分类器适应新数据：删除顶层，添加新的分类层，并仅在该层上进行训练
* 冻结层时，避免跟踪统计数据（例如在 Batch Normalization 层中）
* 微调模型的最后几层，以捕获网络末端附近的高级细节，并有可能提高准确性

</div>

## 恭喜！

你已经完成了关于迁移学习的作业和微调。以下是你刚刚完成的所有工作的快速回顾：

* 从目录创建数据集
* 使用 Sequential API 增强数据
* 使用Functional API 和 MobileNetV2 将预训练模型调整为新数据
* 微调分类器的最终层并提高模型的准确性

太棒了！ 